In [ ]:
!pip install -q x-transformers
!pip install -q flash-attn --no-build-isolation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import os
import sys
import subprocess
import hashlib
import gc
import platform
from datetime import datetime
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import RobertaTokenizerFast, get_cosine_schedule_with_warmup, DataCollatorForLanguageModeling
from datasets import load_dataset
from x_transformers import Encoder

# ==========================================
# 1. CONFIGURATION
# ==========================================
# YOUR REPO ID (Created in previous step)
HF_ID = "prism-lab/wikitext-103-prism-32k-seq4k"

# Hyperparameters
VOCAB_SIZE = 32768
SEQ_LEN = 4096
BATCH_SIZE = 8
EPOCHS = 40
LR = 1e-3
D_MODEL = 512
RESUME_PATH = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# ==========================================
# 2. DATA PIPELINE (The "Pro" Way)
# ==========================================
def prepare_data_from_hub():
    print(f"⬇️ Pulling Pre-Tokenized Data from {HF_ID}...")

    # 1. Load Tokenizer (Instant)
    # This pulls the exact tokenizer you uploaded
    tokenizer = RobertaTokenizerFast.from_pretrained(HF_ID)

    # 2. Load Dataset (Instant)
    # This pulls the already chunked/tokenized data
    dataset = load_dataset(HF_ID)

    print(f"✅ Loaded {len(dataset['train'])} training chunks.")

    # 3. Collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    return dataset, data_collator


class FNetBlock(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm_mix = nn.LayerNorm(d_model) # LayerNorm is safer for FNet than RMSNorm
        self.norm_ff = nn.LayerNorm(d_model)

        self.mix_dropout = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # 1. Fourier Mixing Branch
        residual = x
        x = self.norm_mix(x)

        # --- THE FIX ---
        with torch.cuda.amp.autocast(enabled=False):
            x = x.float()
            # norm='ortho' makes the FFT energy-preserving.
            # Output magnitude will match input magnitude (~1).
            x = torch.fft.fftn(x, dim=(-2, -1), norm='ortho').real
            x = x.to(dtype=residual.dtype)
        # ---------------

        # Now 'x' and 'residual' have roughly same magnitude.
        # The skip connection works again.
        x = self.mix_dropout(x)
        x = x + residual

        # 2. Feed Forward Branch
        residual = x
        x = self.norm_ff(x)
        x = self.ff(x)
        return x + residual


class FNetEncoder(nn.Module):
    def __init__(self, depth, d_model, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            FNetBlock(d_model, d_ff, dropout) for _ in range(depth)
        ])
        # [FIX] Use LayerNorm here to match the blocks
        self.norm_out = nn.LayerNorm(d_model)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm_out(x)


class HybridFNetMLM(nn.Module):
    def __init__(self, vocab_size, d_model, seq_len, d_ff, dropout):
        super().__init__()

        # 1. Standard Embeddings + Absolute Positions
        # (FNet NEEDS these because FFT is position-blind)
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)

        self.fnet_encoder = FNetEncoder(
            depth=6,
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        # 3. The Attention Cap (1 Layer) -> YOUR CONFIGURATION
        self.transformer_cap = Encoder(
            dim=d_model,
            depth=1,                # Just 1 layer
            heads=8,
            rotary_pos_emb=True,    # RoPE (Hybrid Positioning: Absolute for FNet, Rotary for Attn)
            attn_flash=True,
            attn_dropout=dropout,
            ff_dropout=dropout
            # Removed 'dim_head' (fixes your error)
            # Removed 'use_rmsnorm' (matches your snippet)
            # Removed 'ff_glu' (matches your snippet)
        )

        # 4. MLM Head
        self.final_norm = nn.LayerNorm(d_model)
        self.to_logits = nn.Linear(d_model, vocab_size)

        # Weight Tying
        self.to_logits.weight = self.token_emb.weight

    def forward(self, input_ids):
        # A. Embedding
        x = self.token_emb(input_ids)
        b, n, d = x.shape

        # Add Absolute Positions (Crucial for FNet layers)
        x = x + self.pos_emb[:, :n, :]
        x = self.dropout(x)

        x = self.fnet_encoder(x)

        # C. Attention Refinement (1 Layer)
        # Note: This layer will internally apply RoPE to Q/K
        x = self.transformer_cap(x)

        # D. Output
        x = self.final_norm(x)
        return self.to_logits(x)

# ==========================================
# INSTANTIATE MODEL
# ==========================================
print("🏗️  Constructing Hybrid FNet (6-Spectral + 1-Attention)...")

def count_active_parameters(model):
    print(f"\n{'='*60}")
    print(f"🧩 DETAILED PARAMETER BREAKDOWN")
    print(f"{'='*60}")

    # 1. Identify Parameter Groups
    # ----------------------------
    embedding_ids = set()
    active_ids = set()
    unique_params = set()

    # --- MEMORY (Embeddings & Encodings) ---
    embedding_params = 0
    for p in model.token_emb.parameters():
        embedding_params += p.numel()
        embedding_ids.add(id(p))
        unique_params.add(id(p))

    pos_params = model.pos_emb.numel()
    embedding_ids.add(id(model.pos_emb))
    unique_params.add(id(model.pos_emb))

    total_memory = embedding_params + pos_params

    # --- LOGIC (Active Processing) ---
    active_count = 0
    for name, param in model.named_parameters():
        if id(param) in embedding_ids:
            continue
        if id(param) not in active_ids:
            active_count += param.numel()
            active_ids.add(id(param))
            unique_params.add(id(param))

    # 2. Calculate Totals
    # -------------------
    total_physical_params = total_memory + active_count

    # 3. Print Report (FIXED SYNTAX)
    # -------------------
    print(f"{'Component':<25} | {'Count':<15} | {'% of Model':<10}")
    print(f"{'-'*60}")

    print(f"{'Token Embeddings':<25} | {embedding_params:<15,} | {embedding_params/total_physical_params:.1%}")
    print(f"{'Positional Encodings':<25} | {pos_params:<15,} | {pos_params/total_physical_params:.1%}")
    print(f"{'[MEMORY TOTAL]':<25} | {total_memory:<15,} | {total_memory/total_physical_params:.1%}")
    print(f"{'-'*60}")

    fnet_params = sum(p.numel() for p in model.fnet_encoder.parameters())
    cap_params = sum(p.numel() for p in model.transformer_cap.parameters())
    misc_params = active_count - fnet_params - cap_params

    print(f"{'FNet Encoder (6 Layers)':<25} | {fnet_params:<15,} | {fnet_params/total_physical_params:.1%}")
    print(f"{'Transformer Cap (1 Layer)':<25}| {cap_params:<15,} | {cap_params/total_physical_params:.1%}")
    print(f"{'Norms & Biases':<25} | {misc_params:<15,} | {misc_params/total_physical_params:.1%}")
    print(f"{'[ACTIVE LOGIC TOTAL]':<25} | {active_count:<15,} | {active_count/total_physical_params:.1%}")

    print(f"{'='*60}")
    print(f"📢 FINAL ACTIVE PARAMETERS: {active_count / 1_000_000:.2f} M")
    print(f"{'='*60}\n")

    return active_count




In [ ]:
# ==========================================
# 3. INITIALIZATION FUNCTION (FNet Specific)
# ==========================================
def init_fnet_weights(model):
    print("✨ Applying BERT-Style Initialization (N(0, 0.02))...")

    for name, module in model.named_modules():
        # A. Linear Layers (Projections, FFNs)
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if module.bias is not None:
                module.bias.data.zero_()

        # B. Embeddings (Tokens)
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

        # C. LayerNorms (Stability)
        elif isinstance(module, nn.LayerNorm):
            if module.bias is not None:  # <--- FIX IS HERE
                module.bias.data.zero_()
            if module.weight is not None:
                module.weight.data.fill_(1.0)

    # D. Positional Embeddings (Manually handle the nn.Parameter)
    if hasattr(model, 'pos_emb') and model.pos_emb is not None:
        model.pos_emb.data.normal_(mean=0.0, std=0.02)

    print("✅ Initialization Complete.")


# ==========================================
# 4. LOGGING UTILITIES
# ==========================================
def generate_run_id():
    raw = datetime.now().strftime("%Y%m%d%H%M%S%f")
    return hashlib.md5(raw.encode()).hexdigest()[:8]

def log_full_environment(save_dir, run_id, config):
    log_path = os.path.join(save_dir, f"env_metadata_{run_id}.txt")

    # 1. Gather System Info
    sys_info = {
        "Python Version": sys.version.split()[0],
        "OS": platform.platform(),
        "PyTorch Version": torch.__version__,
        "CUDA Available": torch.cuda.is_available(),
        "CUDNN Version": torch.backends.cudnn.version() if torch.cuda.is_available() else "N/A"
    }

    # 2. Gather GPU Info
    gpu_info = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            gpu_info.append(f"GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB")
    else:
        gpu_info.append("No GPU Detected")

    # 3. Gather Pip Freeze
    try:
        pip_packages = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze']).decode('utf-8')
    except Exception as e:
        pip_packages = f"Could not retrieve pip packages: {e}"

    # 4. Write to File
    with open(log_path, "w") as f:
        f.write(f"🧪 EXPERIMENT METADATA | Run ID: {run_id}\n")
        f.write(f"{'='*60}\n\n")

        f.write(f"--- [1] CONFIGURATION ---\n")
        for k, v in config.items():
            f.write(f"{k}: {v}\n")
        f.write("\n")

        f.write(f"--- [2] SYSTEM HARDWARE ---\n")
        for k, v in sys_info.items():
            f.write(f"{k}: {v}\n")
        for g in gpu_info:
            f.write(f"{g}\n")
        f.write("\n")

        f.write(f"--- [3] INSTALLED PACKAGES (pip freeze) ---\n")
        f.write(pip_packages)

    print(f"📝 Full Environment Snapshot (GPU + Pip) saved to: {log_path}")


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, config):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_loss,
        'config': config
    }, path)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def run_wikitext_training(experiment_name="FNet_Encoder"):
    from google.colab import drive
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')

    # --- SETUP DIRS ---
    if RESUME_PATH and os.path.exists(RESUME_PATH):
        print(f"🔄 RESUMING FROM: {RESUME_PATH}")
        checkpoint = torch.load(RESUME_PATH, map_location=DEVICE)
        SAVE_DIR = os.path.dirname(RESUME_PATH)
        run_id = checkpoint.get('config', {}).get('run_id', 'resumed')
    else:
        run_id = generate_run_id()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        folder_name = f"{experiment_name}_{timestamp}_{run_id}"
        SAVE_DIR = os.path.join("/content/drive/My Drive/PRISM_Experiments", folder_name)
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"💾 Checkpoints: {SAVE_DIR}")

    writer = SummaryWriter(log_dir=SAVE_DIR)
    GRAD_ACCUM = 4

    # Load Data
    lm_datasets, data_collator = prepare_data_from_hub()

    # Create Loaders
    train_loader = DataLoader(
        lm_datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=data_collator, num_workers=2, pin_memory=True,
        prefetch_factor=2, persistent_workers=True
    )
    valid_loader = DataLoader(
        lm_datasets["validation"], batch_size=BATCH_SIZE,
        collate_fn=data_collator, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        lm_datasets["test"], batch_size=BATCH_SIZE,
        collate_fn=data_collator, num_workers=2, pin_memory=True
    )

    print("\n⚡ INITIALIZING HYBRID FNET MODEL...")

    # INSTANTIATE
    model = HybridFNetMLM(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        seq_len=SEQ_LEN,
        d_ff=D_MODEL * 4,
        dropout=0.1
    ).to(DEVICE)

    # OPTIMIZER
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()

    start_epoch = 0
    best_val_loss = float('inf')

    # RESUME OR INIT
    if RESUME_PATH and os.path.exists(RESUME_PATH):
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        del checkpoint
        torch.cuda.empty_cache()
    else:
        # [UPDATED] CALL THE FNET INITIALIZATION
        init_fnet_weights(model)

    # METRICS
    try:
        active_params = count_active_parameters(model) # Uses function defined in prev step
    except:
        print("⚠️ Parameter counter not found, skipping detailed breakdown.")

    total_params = sum(p.numel() for p in model.parameters())
    print(f"✅ Model Ready. Total Raw Params: {total_params/1e6:.2f}M")

    log_full_environment(SAVE_DIR, run_id, {
        "model": "HybridFNetMLM",
        "d_model": D_MODEL,
        "depth": "6+1",
        "vocab": VOCAB_SIZE,
        "batch": BATCH_SIZE,
        "lr": LR,
        "active_params": f"{active_params/1e6:.2f}M"
    })

    print(f"\n🚀 STARTING (Ep {start_epoch+1} to {EPOCHS})")
    global_step = (len(train_loader) // GRAD_ACCUM) * start_epoch

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar):
            x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)

            # FNet Forward Pass
            logits = model(x)

            # Loss Calculation
            loss = criterion(logits.view(-1, VOCAB_SIZE), y.view(-1)) / GRAD_ACCUM
            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                # 1. Calc Norm
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                # 2. Step
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                # 3. LOGGING
                actual_loss = loss.item() * GRAD_ACCUM
                writer.add_scalar('Train/Loss', actual_loss, global_step)
                writer.add_scalar('Train/GradNorm', grad_norm.item(), global_step)
                writer.add_scalar('Train/LR', scheduler.get_last_lr()[0], global_step)

                # 4. Progress Bar
                pbar.set_postfix({
                    'loss': f"{actual_loss:.4f}",
                    'gnorm': f"{grad_norm.item():.2f}"
                })

        # VALIDATION
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in valid_loader:
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                val_loss += criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)).item()

        avg_val_loss = val_loss / len(valid_loader)
        ppl = math.exp(avg_val_loss) if avg_val_loss < 100 else float('inf')

        print(f"✨ Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | PPL: {ppl:.2f}")
        writer.add_scalar('Val/PPL', ppl, epoch+1)
        writer.add_scalar('Val/Loss', avg_val_loss, epoch+1)

        config_dump = {"epoch": epoch, "run_id": run_id}
        save_checkpoint(os.path.join(SAVE_DIR, "last.pt"), model, optimizer, scheduler, epoch, best_val_loss, config_dump)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best.pt"))
            print("   🏆 New Best Model Saved!")

    # FINAL TEST
    best_path = os.path.join(SAVE_DIR, "best.pt")
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path))
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Testing"):
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                test_loss += criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)).item()
        print(f"🏆 FINAL TEST PPL: {math.exp(test_loss/len(test_loader)):.2f}")

    writer.close()
    return model

In [ ]:
if __name__ == "__main__":


    # 1. Run the Training Routine
    # This handles Model Creation -> Analysis -> Training -> Saving
    trained_prism = run_wikitext_training()

    # 2. Cleanup & Shutdown


In [ ]:
from google.colab import runtime
runtime.unassign()